In [2]:
import os
import re

In [ ]:

# Ruta a la carpeta donde están los archivos
carpeta = 'SAFE_downloads'

# Expresión regular para capturar la fecha después de "MSIL1C_"
regex = re.compile(r'MSIL1C_(\d{8})T')

fechas = []

# Recorremos los archivos de la carpeta
for archivo in os.listdir(carpeta):
    if archivo.endswith('.zip'):
        match = regex.search(archivo)
        if match:
            fecha = match.group(1)  # formato YYYYMMDD
            # Convertimos a formato YYYY-MM-DD
            fechas.append(f"{fecha[:4]}-{fecha[4:6]}-{fecha[6:]}")

# Resultado
print(sorted(fechas))


### Comprobar nubosidad para una fecha dada

In [3]:
from sentinelhub import SentinelHubCatalog, DataCollection, BBox, CRS, bbox_to_dimensions, SHConfig
from datetime import datetime
import configparser
from utils import get_access_token
from datetime import datetime, timedelta

/home/antonio/.pyenv/versions/3.8.5/envs/nitrates/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
config_file = configparser.ConfigParser()
config_file.read("config.ini")

username = config_file["copernicus"]["username"]
password = config_file["copernicus"]["password"]

config = SHConfig()
config.sh_client_id = config_file["copernicus"]["client_id"] #"<CLIENT ID>"
config.sh_client_secret = config_file["copernicus"]["client_secret"] #<CLIENT SECRET>"
config.sh_token_url = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token" # Is it required?
config.sh_base_url = "https://sh.dataspace.copernicus.eu"
config.save("cdse")
config = SHConfig("cdse")

In [5]:

def check_cloud_cover(access_token, time_interval, aoi, resolution, config): 
    # Definir el área de interés
    aoi_bbox = BBox(bbox=aoi, crs=CRS.WGS84)

    # Inicializar el catálogo
    catalog = SentinelHubCatalog(config=config)
    date_range = time_interval[0], time_interval[1]

    # Buscar imágenes en el intervalo de fechas
    search_iterator = catalog.search(
        DataCollection.SENTINEL2_L2A,
        bbox=aoi_bbox,
        time=date_range,
        fields={"include": ["id", "properties.eo:cloud_cover"], "exclude": ["properties.datetime"]},
    )
    results = list(search_iterator)

    # Filtrar resultados únicos por ID
    unique_results = {}
    for item in results:
        acquisition_id = item['id'].split('_T')[0]
        if acquisition_id not in unique_results:
            unique_results[acquisition_id] = item

    # Mostrar la nubosidad por fecha
    for item in unique_results.values():
        image_id = item['id']
        date = datetime.strptime(image_id.split('_')[2][:8], "%Y%m%d").date()
        cloud_cover = item["properties"]["eo:cloud_cover"]
        if cloud_cover > 20:
            print(f"🌩️ -- {date} - Cloud cover: {cloud_cover}%")
        else:
            print(f"     {date} - Cloud cover: {cloud_cover}%")


In [7]:
access_token = get_access_token(username, password)

# Fechas que tenemos
# target_dates = [
#     '2016-08-09', '2016-09-08', '2016-10-28', '2017-06-30', '2018-02-20', '2018-03-07', 
#     '2018-05-16', '2018-06-20', '2018-07-10', '2018-08-29', '2018-10-03',
#     '2018-11-07', '2019-03-12', '2019-06-25', '2019-07-10', '2019-07-30', 
#     '2019-08-14', '2019-09-18', '2019-09-28', '2019-10-03', '2019-11-27', '2020-02-20', 
#     '2020-03-11', '2020-12-21', '2021-01-05', '2021-04-20', '2021-05-20',
#     '2021-06-14', '2021-08-13', '2021-11-11', '2021-11-26', '2021-12-01', 
#     '2022-02-24', '2022-07-14', '2022-07-29', '2022-08-03', '2022-09-07', 
#     '2022-09-27', '2023-01-10', '2023-03-01', '2023-03-16', '2023-04-20', 
#     '2023-05-25']

# Fechas de los informes de IMIDA
target_dates = [
    '2021-08-12', '2021-08-17', '2021-08-20', '2021-08-21', '2021-08-22',
    '2021-08-25', '2021-08-27', '2021-08-28', '2021-08-29', '2021-08-30',
    '2021-09-02', '2021-09-03', '2021-09-04', '2021-09-05', '2021-09-06',
    '2021-09-07', '2021-09-08', '2021-09-09', '2021-09-10', '2021-09-11',
    '2021-09-12', '2021-09-13', '2021-09-14', '2021-09-15', '2021-09-20',
    '2021-09-27', '2021-09-29', '2021-10-02', '2021-10-04', '2021-10-06',
    '2021-10-08', '2021-10-13', '2021-10-15', '2021-10-18', '2021-10-25',
    '2021-10-27', '2021-11-02', '2021-11-08', '2021-11-15', '2021-11-22',
    '2021-11-30', '2021-12-07', '2021-12-13', '2021-12-20', '2021-12-28',
    '2022-01-03', '2022-01-12', '2022-01-17', '2022-01-24', '2022-01-31',
    '2022-02-07', '2022-02-21', '2022-03-01', '2022-03-08', '2022-04-06',
    '2022-04-18', '2022-04-25', '2022-05-03', '2022-05-09', '2022-05-16',
    '2022-05-23', '2022-05-30', '2022-06-06', '2022-06-13', '2022-06-18',
    '2022-06-27', '2022-07-11', '2022-07-18', '2022-07-25', '2022-08-01',
    '2022-08-08', '2022-08-30', '2022-09-12', '2022-09-20', '2022-09-27',
    '2022-10-04', '2022-10-10', '2022-10-19', '2022-11-02', '2022-11-08',
    '2022-11-16', '2022-12-19', '2022-12-28', '2023-01-05', '2023-01-12',
    '2023-01-27', '2023-02-01', '2023-02-14', '2023-02-23', '2023-03-06',
    '2023-03-16', '2023-03-21', '2023-03-22', '2023-04-20', '2023-04-27',
    '2023-05-04', '2023-05-18', '2023-05-25', '2023-06-05', '2023-06-06',
    '2023-11-28', '2023-12-04', '2023-12-19', '2023-12-26', '2024-01-03',
    '2024-01-09', '2024-01-16', '2024-01-23', '2024-02-01', '2024-02-06',
    '2024-02-13', '2024-02-19', '2024-03-02', '2024-03-06', '2024-03-13',
    '2024-03-18', '2024-04-03', '2024-04-10', '2024-04-15', '2024-04-22',
    '2024-04-30', '2024-05-06', '2024-05-13', '2024-06-17', '2024-06-25',
    '2024-07-01', '2024-07-09', '2024-07-15', '2024-07-22', '2024-07-31',
    '2024-08-13', '2024-08-20', '2024-08-27', '2024-09-03', '2024-09-10',
    '2024-09-19', '2024-09-24', '2024-10-01', '2024-10-08', '2024-10-15',
    '2024-10-22', '2024-10-30', '2024-11-05', '2024-11-12', '2024-11-18',
    '2024-11-25', '2024-12-03', '2024-12-17', '2024-12-26', '2025-01-08',
    '2025-01-13', '2025-01-20', '2025-01-29', '2025-02-05', '2025-02-10',
    '2025-02-17', '2025-02-24', '2025-03-07', '2025-03-17', '2025-03-25',
    '2025-03-31', '2025-04-08', '2025-04-14', '2025-04-21', '2025-04-28'
]


slots = [((datetime.strptime(d, "%Y-%m-%d") - timedelta(days=1)).strftime("%Y-%m-%d"), (datetime.strptime(d, "%Y-%m-%d") + timedelta(days=4)).strftime("%Y-%m-%d")) for d in target_dates]
# Para comprobar solamente el día que interesa
slots = [(d, d) for d in target_dates]

for time_interval in slots: 

    check_cloud_cover(
        access_token=access_token,
        time_interval=time_interval,
        aoi= [-0.86, 37.65, -0.74, 37.8],  # coordenadas de ejemplo (W, S, E, N)
        resolution=10,
        config=config
    )

     2021-08-28 - Cloud cover: 5.5%
🌩️ -- 2021-09-02 - Cloud cover: 46.69%
🌩️ -- 2021-09-07 - Cloud cover: 30.64%
     2021-09-12 - Cloud cover: 0.0%
🌩️ -- 2021-09-27 - Cloud cover: 84.3%
🌩️ -- 2021-10-02 - Cloud cover: 39.44%
🌩️ -- 2021-10-27 - Cloud cover: 55.17%
     2022-03-01 - Cloud cover: 0.04%
     2022-04-25 - Cloud cover: 2.26%
🌩️ -- 2022-05-30 - Cloud cover: 21.06%
     2022-08-08 - Cloud cover: 12.93%
🌩️ -- 2022-09-12 - Cloud cover: 75.47%
     2022-09-27 - Cloud cover: 0.35%
     2022-09-27 - Cloud cover: 0.36%
🌩️ -- 2022-11-16 - Cloud cover: 100.0%
🌩️ -- 2022-11-16 - Cloud cover: 100.0%
🌩️ -- 2023-01-05 - Cloud cover: 27.69%
🌩️ -- 2023-01-05 - Cloud cover: 27.68%
🌩️ -- 2023-02-14 - Cloud cover: 99.02%
🌩️ -- 2023-02-14 - Cloud cover: 99.02%
🌩️ -- 2023-03-06 - Cloud cover: 91.75%
🌩️ -- 2023-03-06 - Cloud cover: 91.74%
     2023-03-16 - Cloud cover: 0.0%
     2023-03-16 - Cloud cover: 0.0%
🌩️ -- 2023-03-21 - Cloud cover: 71.65%
🌩️ -- 2023-03-21 - Cloud cover: 71.68%
     202

In [1]:
fechas_descargadas = [
    '2016-08-09', '2016-09-08', '2016-10-28', '2017-01-26', '2017-06-30', 
    '2017-07-05', '2017-11-22', '2018-01-31', '2018-02-20', '2018-03-07', 
    '2018-05-11', '2018-05-16', '2018-06-20', '2018-07-10', '2018-08-09', 
    '2018-08-14', '2018-08-29', '2018-09-13', '2018-10-03', '2018-11-07', 
    '2019-02-20', '2019-03-07', '2019-03-12', '2019-06-25', '2019-07-10', 
    '2019-07-30', '2019-08-14', '2019-08-29', '2019-09-18', '2019-09-28', 
    '2019-10-03', '2019-11-07', '2019-11-27', '2019-12-02', '2019-12-07',
    '2020-02-20', '2020-02-25', '2020-03-11', '2020-05-05', '2020-05-20',
    '2020-05-25', '2020-12-21', '2021-01-05', '2021-01-20', '2021-02-24', 
    '2021-04-20', '2021-05-20', '2021-05-25', '2021-06-14', '2021-08-03',
    '2021-08-13', '2021-09-02', '2021-09-27', '2021-10-07', '2021-11-11', 
    '2021-11-26', '2021-12-01', '2021-12-21', '2022-02-24', '2022-03-11', 
    '2022-06-24', '2022-07-14', '2022-07-29', '2022-08-03', '2022-09-07', 
    '2022-09-27', '2023-01-10', '2023-01-20', '2023-01-25', '2023-02-14', 
    '2023-03-01', '2023-03-16', '2023-04-20', '2023-05-05', '2023-05-25', 
    '2023-07-19', '2023-09-07', '2023-09-27', '2023-10-17', '2023-11-16'
]

fechas_filtro_1 = [
    '2018-01-31', '2018-08-09', '2018-08-14', '2019-03-07', '2019-09-18',
    '2019-11-07', '2020-02-25', '2020-05-20', '2020-05-05', '2021-07-14', 
    '2021-09-27', '2022-06-24', '2023-01-20', '2023-07-19', '2023-09-07', 
    '2023-09-27', '2023-11-16'
]

fechas_filtro_2 = [
    '2016-10-28', '2017-01-26', '2017-07-05', '2017-11-22', '2018-01-31', 
    '2018-05-11', '2018-09-13', '2019-02-20', '2019-03-07', '2019-08-29',
    '2019-09-18', '2019-11-07', '2019-12-02', '2019-12-07', '2020-02-25',
    '2020-05-05', '2020-05-25', '2021-01-20', '2021-02-24', '2021-05-25',
    '2021-07-14', '2021-08-03', '2021-09-02', '2021-09-27', '2021-10-07',
    '2021-12-21', '2022-03-11', '2022-06-24', '2023-01-20', '2023-01-25',
    '2023-02-14', '2023-05-05', '2023-07-19', '2023-09-07', '2023-09-27', 
    '2023-10-17', '2023-11-16'
]

In [2]:
fechas_disponibles = set(fechas_descargadas) - set(fechas_filtro_1) - set(fechas_filtro_2)

In [3]:
list(fechas_disponibles).sort()
print(len(fechas_disponibles))

41
